# Chapter 2 — Working with Text Data

Building the data pipeline for an LLM: downloading text, tokenizing, building a dataset/dataloader, and creating token embeddings.

## 1. Download the dataset

We use *The Verdict* by Edith Wharton as our raw training text.

In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    print("Total number of character:", len(raw_text))
    print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2. A simple tokenizer

`SimpleTokenizerV1` maps words to integer IDs and back using a vocabulary.

In [2]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = text.split(" ")
        encoded_tokens = [self.str_to_int[token] for token in tokens]
        return encoded_tokens

    def decode(self, tokens):
        return " ".join([self.int_to_str[token] for token in tokens])

### Handling unknown tokens

`SimpleTokenizerV2` falls back to an `<unk>` token for words not in the vocabulary.

In [3]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        self.unknown_token = self.str_to_int["<unk>"]

    def encode(self, text):
        tokens = text.split(" ")
        encoded_tokens = [
            self.str_to_int.get(token, self.unknown_token) for token in tokens
        ]
        return encoded_tokens

    def decode(self, tokens):
        return " ".join([self.int_to_str[token] for token in tokens])

### Additional special tokens

- **`[BOS]`** (beginning of sequence) — marks the start of a text; signifies to the LLM where a piece of content begins.
- **`[EOS]`** (end of sequence) — positioned at the end of a text, useful when concatenating multiple unrelated texts (similar to `<|endoftext|>`). For instance, when combining two different Wikipedia articles or books, the `[EOS]` token indicates where one ends and the next begins.
- **`[PAD]`** (padding) — when training with batch sizes larger than one, the batch might contain texts of varying lengths. To ensure all texts have the same length, shorter texts are extended ("padded") using the `[PAD]` token, up to the length of the longest text in the batch.

## 3. Byte pair encoding (BPE)

Instead of our toy tokenizer, we use GPT-2's BPE tokenizer from `tiktoken`. Below we also demonstrate the input/target relationship: given a context `x[:i]`, the next token is `y[i-1]`.

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

CONTEXT_SIZE = 4
x = enc_text[:CONTEXT_SIZE]
y = enc_text[1 : CONTEXT_SIZE + 1]
for i in range(1, CONTEXT_SIZE + 1):
    print(tokenizer.decode(x[:i]), ">>", tokenizer.decode([y[i - 1]]))

5145
I >>  H
I H >> AD
I HAD >>  always
I HAD always >>  thought


## 4. Dataset and DataLoader

`TextDataset` slides a window of `max_length` tokens over the text. **X** is each window; **Y** is the same window shifted by one position. `unfold(dim, size, step)` builds all windows at once.

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, tokenizer, text, max_length, stride):
        token_ids = torch.tensor(tokenizer.encode(text))

        # X is each window of `max_length` tokens; Y is the same window
        # shifted by one. unfold(dim, size, step) builds all windows at once.
        self.input_ids = token_ids[:-1].unfold(0, max_length, stride)
        self.target_ids = token_ids[1:].unfold(0, max_length, stride)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [6]:
def create_dataloader_v1(text, batch_size, max_length, stride, shuffle, drop_last, num_workers):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = TextDataset(tokenizer, text, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader


dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=10,
    stride=1,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


## 5. Token embeddings

An `nn.Embedding` is a trainable lookup table that maps each token ID to a vector. With `VOCAB_SIZE=50257` (GPT-2 BPE) and `OUTPUT_DIM=256`, a batch of shape `[batch_size, max_length]` becomes `[batch_size, max_length, OUTPUT_DIM]`.

In [7]:
VOCAB_SIZE = 50257  # GPT-2 BPE vocabulary size
OUTPUT_DIM = 256  # length of each token's embedding vector
token_embedding_layer = torch.nn.Embedding(VOCAB_SIZE, OUTPUT_DIM)

inputs, targets = first_batch
print("Input shape:", inputs.shape)  # [batch_size, max_length]

token_embeddings = token_embedding_layer(inputs)
print("Token embeddings shape:", token_embeddings.shape)  # [batch_size, max_length, OUTPUT_DIM]

Input shape: torch.Size([1, 10])
Token embeddings shape: torch.Size([1, 10, 256])
